In [ ]:
!pip install datasets spacy pycountry pandas tqdm
!python -m spacy download en_core_web_trf  


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 21.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.7/656.7 kB 33.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.1/788.1 kB 32.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 36.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [spacy]m19/20 [spacy]d]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 53.1 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 48.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for curated-tokenizers: filename=curated_tokenizers-0.0.9-cp313-cp313-macosx_11_0_arm64.whl size=911695 sha256=989e428b7626af7923736f311f589a48163aff9dd6b8f1457d9843320e2cfbbe
  Stored in directory: /Users/karo

In [4]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 54.6 MB/s  0:00:06:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [5]:
import spacy
import pycountry
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from collections import Counter

MODEL = "en_core_web_lg"
MAX_ARTICLES = 5000
BATCH_SIZE = 64

# Codes to always discard — known false positives
BLOCKLIST = {"VI", "GE", "LR", "MC", "GS", "UM", "TF", "IO", "SH"}

# Minimum character length for a mention to be trusted
MIN_MENTION_LENGTH = 4

CUSTOM_MAP = {
    # Abbreviations
    "US": "United States", "USA": "United States", "America": "United States",
    "UK": "United Kingdom", "Britain": "United Kingdom", "England": "United Kingdom",
    "UAE": "United Arab Emirates",
    # Official name mismatches
    "Russia": "Russian Federation", "Iran": "Iran, Islamic Republic of",
    "Syria": "Syrian Arab Republic", "North Korea": "Korea, Democratic People's Republic of",
    "South Korea": "Korea, Republic of", "Vietnam": "Viet Nam",
    "Bolivia": "Bolivia, Plurinational State of", "Venezuela": "Venezuela, Bolivarian Republic of",
    # Demonyms
    "American": "United States", "British": "United Kingdom", "French": "France",
    "German": "Germany", "Chinese": "China", "Russian": "Russian Federation",
    "Iraqi": "Iraq", "Afghan": "Afghanistan", "Iranian": "Iran, Islamic Republic of",
    "Israeli": "Israel", "Pakistani": "Pakistan", "Indian": "India",
    "Australian": "Australia", "Canadian": "Canada", "Mexican": "Mexico",
    "Colombian": "Colombia", "Brazilian": "Brazil", "Saudi": "Saudi Arabia",
    # Explicit suppressions
    "Georgia": None, "Virgin": None, "Virgin Islands": None,
    "Liberia": None,   # too many false positives — add back if you need it
    "Civil": None, "Monaco": None,
}

def resolve_country(mention):
    mention = mention.strip()

    # drop short mentions — single words under 4 chars are too ambiguous
    if len(mention) < MIN_MENTION_LENGTH:
        return None

    # check custom map
    if mention in CUSTOM_MAP:
        mapped = CUSTOM_MAP[mention]
        if mapped is None:
            return None
        mention = mapped

    # exact lookup
    try:
        c = pycountry.countries.lookup(mention)
        if c.alpha_2 in BLOCKLIST:
            return None
        return c.alpha_2
    except LookupError:
        pass

    # fuzzy lookup — only trust it if the mention is long enough to be meaningful
    if len(mention) >= 6:
        try:
            c = pycountry.countries.search_fuzzy(mention)[0]
            if c.alpha_2 in BLOCKLIST:
                return None
            return c.alpha_2
        except LookupError:
            pass

    return None


nlp = spacy.load(MODEL)

ds = load_dataset("cnn_dailymail", "3.0.0", split="train")
ds = ds.select(range(MAX_ARTICLES))

texts = [row["article"] for row in ds]
ids   = [row["id"]      for row in ds]

results = []
for i, doc in enumerate(tqdm(
    nlp.pipe(texts, batch_size=BATCH_SIZE, disable=["parser", "senter"]),
    total=len(texts)
)):
    counts = Counter()
    for ent in doc.ents:
        if ent.label_ in ("GPE", "LOC"):
            code = resolve_country(ent.text)
            if code:
                counts[code] += 1

    primary = counts.most_common(1)[0][0] if counts else None
    results.append({
        "id": ids[i],
        "primary_country": primary,
        "country_freq": dict(counts)
    })

df = pd.DataFrame(results)
df.to_parquet("ner_results.parquet", index=False)
print(df["primary_country"].value_counts().head(20))

100%|██████████| 5000/5000 [08:02<00:00, 10.37it/s]

primary_country
US    1959
GB     485
IQ     208
PK     119
CN     106
IN     106
MX     101
SO      84
AF      71
IR      67
ES      65
IL      60
DE      54
FR      54
IT      50
ZW      49
RU      47
ZA      40
JP      39
LK      36
Name: count, dtype: int64


,id,primary_country,country_freq
0,42c027e4ff9730fbb3de84c1af0d2c506e41c3e4,GB,"{'GB': 3, 'CO': 1}"
1,ee8871b15c50d0db17b0179a6d2beab35065f1e9,US,{'US': 1}
2,06352019a19ae31e527f37f7571c6dd7f0c5da37,US,{'US': 2}
3,24521a2abb2e1f5e34e6824e0f9e56904a2b0e88,LR,"{'US': 2, 'LR': 3, 'ZW': 1}"
4,7fe70cc8b12fab2d0a258fababf7d9c6b5e1262a,US,"{'US': 8, 'GB': 2, 'GE': 1}"
5,a1ebb8bb4d370a1fdf28769206d572be60642d70,IQ,"{'IQ': 5, 'US': 2, 'VI': 1}"
6,7c0e61ac829a3b3b653e2e3e7536cc4881d1f264,IQ,{'IQ': 7}
7,f0d73bdab711763e745cdc75850861c9018f235d,VI,"{'CO': 5, 'VI': 7, 'VE': 1}"
8,5e22bbfc7232418b8d2dd646b952e404df5bd048,US,{'US': 1}
9,613d6311ec2c1985bd44707d1796d275452fe156,US,{'US': 2}


In [ ]:
import random

# pick 20 random indices from outside the first 5000 we already processed
random_indices = random.sample(range(5000, 50000), 250)

ds_sample = load_dataset("cnn_dailymail", "3.0.0", split="train")
ds_sample = ds_sample.select(random_indices)

texts = [row["article"] for row in ds_sample]
ids   = [row["id"]      for row in ds_sample]

results_sample = []
for i, doc in enumerate(tqdm(
    nlp.pipe(texts, batch_size=20, disable=["parser", "senter"]),
    total=len(texts)
)):
    counts = Counter()
    for ent in doc.ents:
        if ent.label_ in ("GPE", "LOC"):
            code = resolve_country(ent.text)
            if code:
                counts[code] += 1

    primary = counts.most_common(1)[0][0] if counts else None
    results_sample.append({
        "id": ids[i],
        "primary_country": primary,
        "country_freq": dict(counts),
        "article_text": texts[i]
    })

df_sample = pd.DataFrame(results_sample)

for _, row in df_sample.iterrows():
    print("=" * 80)
    print(f"Primary country: {row['primary_country']}")
    print(f"Country freq:    {row['country_freq']}")
    print(f"Article snippet: {row['article_text'][:800]}")
    print()


100%|██████████| 250/250 [00:23<00:00, 10.74it/s]

Primary country: US
Country freq:    {'US': 4, 'NZ': 2, 'WS': 3, 'AS': 1, 'TO': 1}
Article snippet: (CNN) -- When an earthquake threatens to turn part of an ocean into fast-moving walls of water, tsunami warning scientists can do nothing for the first five minutes except wait for information. But within the next five minutes, they have to decide whether to issue a warning of danger. Brian Shiro has been a geophysicist at the Pacific Tsunami Warning Center for four years. And you thought your job was high pressure. "If we see a set of circumstances and it fits into our criteria for [the] event, we just follow that criteria because we don't have much time to think. There isn't a lot of time for decision-making," said Paul Whitmore, director of the West Coast and Alaska Tsunami Warning Center. "Weighing back there [in your mind] also is the effect of your decision. If the effect of your dec

Primary country: ID
Country freq:    {'ID': 12, 'NO': 4}
Article snippet: (CNN) -- One of the worl

In [ ]:
df_sample.to_csv("sample.csv", index=False)
